# 07 — Five training seeds, one fixed split
Run after notebook 05. Reuse the saved `samples.pt` and its exact sequence IDs/class weights. Labels are unchanged.

Only initialization, dropout **masks**, and whole-sequence sampling vary with the seed. The dropout **rate**, architecture, learning rate, epochs, batch size, class weights and sampling weight stay fixed. Each seed selects its best epoch by macro F1, matching notebook 05; ties keep the earlier epoch.

Completed matching seed runs are reused. Interrupted seeds restart from epoch 1. Original `runs/best.pt` and its logs are preserved. Report all five seeds: mean, sample standard deviation (`ddof=1`), minimum, maximum and range. These describe training variability on this validation split, not uncertainty across unseen sequences.


**Padded-box experiment:** retains original DETR features and adds frozen local DINOv2 crop features. Uses this folder's caches, split snapshot, studies and checkpoints.


In [1]:
# Reuse missing base caches in this isolated experiment before checking prerequisites.
from config import reuse_base_caches
reuse_base_caches()

# Cell purpose: Freeze the existing split/settings and configure five independent training seeds.
# Inputs: samples.pt and the original baseline checkpoint settings when available.
# Outputs: Five seeds and one shared parameter dictionary; no split regeneration or training here.

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from config import CONFIG, fingerprint, save_json, require_cache
from training_runs import train_one_seed, summarize_runs, split_manifest, validate_split

# Shared paths come from config.py; they are not hard-coded separately in each notebook.
CACHE = CONFIG['cache_dir']
RUN = CONFIG['run_dir']
print('Sample cache:', CACHE / 'samples.pt')
# Existing samples are authoritative: never rerun splitting for a training seed.
# A missing prerequisite is created by its earlier notebook; an existing cache is reused.
require_cache(CACHE / 'samples.pt', '04_build_training_samples.ipynb')
# Refresh stale padded features before reusing samples/checkpoints.
from config import run_stage
run_stage('04_build_training_samples.ipynb')
data = torch.load(CACHE / 'samples.pt', map_location='cpu', weights_only=True)
# Check disjoint sequence/page groups, class support and original training-only label counts.
validate_split(data)
original = RUN / 'best.pt'
if original.exists():
    checkpoint = torch.load(original, map_location='cpu', weights_only=True)
    # Reject a checkpoint whose sequence/features/split cache differs from the current samples.
    assert checkpoint['samples_signature'] == data['signature'], 'Sample cache changed since baseline; use its matching cache.'
    baseline_config = checkpoint.get('config', CONFIG)
else:
    baseline_config = CONFIG
# Keep the original training settings fixed; the seed experiment changes randomness rather than hyperparameters.
BASE_PARAMS = {
    'lr': float(baseline_config['lr']), 'weight_decay': 0.01, 'dropout': 0.1,
    'batch_size': int(baseline_config['batch_size']), 'epochs': int(baseline_config['epochs']),
    'narration_sampling_weight': float(baseline_config['narration_sampling_weight']),
}
SPLIT = split_manifest(data)
print('Fixed split:', len(SPLIT['train_sequences']), 'train /', len(SPLIT['val_sequences']), 'validation sequences')
print('Original train class counts:', SPLIT['train_counts'])
print('Fixed training settings:', BASE_PARAMS)

# CONFIG: choose five different seeds; use a new RUN_NAME when changing settings.
# Each seed controls initialization, dropout masks and sequence sampling; the saved split stays unchanged.
SEEDS = [42, 123, 2026, 7, 99]
RUN_NAME = 'five_seeds'
REUSE_COMPLETED = True
OUTPUT = RUN / RUN_NAME
assert len(SEEDS) == 5 and len(set(SEEDS)) == 5

Sample cache: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTrainingPaddedBox\cache\samples.pt
Running prerequisite: 04_build_training_samples.ipynb
Running prerequisite: 03a_cache_dino_crops.ipynb
Local encoder: C:\MODELS\dinov2-small
Padding per side: 0.2 | matching feature cache: True
DINO features: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTrainingPaddedBox\cache\dino.pt | sequences: 80
Reuse matching padded samples: True
Exact original split: 64 train / 16 validation
Original train class counts: [1142, 50]
Feature widths: text 768, DETR 256, DINO 384, geometry 4, tail 2; fused input 688.
Saved: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTrainingPaddedBox\cache\samples.pt
Fixed split: 64 train / 16 validation sequences
Original train class counts: [1142, 50]
Fixed training settings: {'lr': 0.0003, 'weight_decay': 0.01, 'dropout': 0.1, 'batch_size': 4, 'epochs': 20, 'narration_s

Train or reuse each seed run and summarize all five best-epoch results.


In [2]:
# Cell purpose: Train or reuse each seed run and summarize all five best-epoch results.
# Inputs: One fixed split/settings manifest and five different training random seeds.
# Outputs: Separate seed artifacts plus mean/std/min/max/range for narration P/R/F1 and macro F1.

OUTPUT.mkdir(parents=True, exist_ok=True)
# Record the exact experiment identity and reject incompatible results on a later rerun.
manifest = {'split': SPLIT, 'params': BASE_PARAMS, 'seeds': SEEDS, 'selection': 'macro_f1'}
manifest_file = OUTPUT / 'manifest.json'
if manifest_file.exists():
    assert json.loads(manifest_file.read_text(encoding='utf-8')) == manifest, 'Settings changed; choose a new RUN_NAME.'
save_json(manifest_file, manifest)

results = []
# Run each seed independently; train_one_seed reuses only a completed matching run.
for seed in SEEDS:
    result = train_one_seed(data, BASE_PARAMS, seed, OUTPUT / f'seed_{seed}', reuse=REUSE_COMPLETED)
    results.append(result)
    save_json(OUTPUT / 'results.json', results)

# Report every seed plus mean, sample std, minimum, maximum and range; do not choose the luckiest seed.
per_seed, summary = summarize_runs(results, OUTPUT)
display(per_seed)
display(summary)
print('Saved:', OUTPUT / 'per_seed_metrics.csv', 'and', OUTPUT / 'seed_summary.csv')

seed 42 epoch 01 | train 0.2411 | val 0.1504 | narration F1 0.000 | macro F1 0.484
seed 42 epoch 02 | train 0.1105 | val 0.1830 | narration F1 0.000 | macro F1 0.487
seed 42 epoch 03 | train 0.0977 | val 0.1459 | narration F1 0.364 | macro F1 0.670
seed 42 epoch 04 | train 0.0175 | val 0.2430 | narration F1 0.118 | macro F1 0.546
seed 42 epoch 05 | train 0.0156 | val 0.2261 | narration F1 0.364 | macro F1 0.670
seed 42 epoch 06 | train 0.0018 | val 0.3110 | narration F1 0.235 | macro F1 0.606
seed 42 epoch 07 | train 0.0067 | val 0.1119 | narration F1 0.533 | macro F1 0.754
seed 42 epoch 08 | train 0.0008 | val 0.3493 | narration F1 0.125 | macro F1 0.550
seed 42 epoch 09 | train 0.0005 | val 0.3753 | narration F1 0.125 | macro F1 0.550
seed 42 epoch 10 | train 0.0002 | val 0.3319 | narration F1 0.400 | macro F1 0.690
seed 42 epoch 11 | train 0.0001 | val 0.3351 | narration F1 0.400 | macro F1 0.690
seed 42 epoch 12 | train 0.0001 | val 0.3315 | narration F1 0.400 | macro F1 0.690
seed

,seed,best_epoch,val_loss,narration_precision,narration_recall,narration_f1,macro_f1
0,42,7,0.111945,0.470588,0.615385,0.533333,0.754299
1,123,7,0.133781,0.538462,0.538462,0.538462,0.758704
2,2026,5,0.068333,0.555556,0.769231,0.645161,0.812846
3,7,8,0.124558,0.500000,0.692308,0.580645,0.778818
4,99,5,0.062639,0.375000,0.692308,0.486486,0.726249


,mean,std,min,max,range
narration_precision,0.487921,0.071285,0.375000,0.555556,0.180556
narration_recall,0.661538,0.087706,0.538462,0.769231,0.230769
narration_f1,0.556818,0.059597,0.486486,0.645161,0.158675
macro_f1,0.766183,0.032129,0.726249,0.812846,0.086598


Saved: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTrainingPaddedBox\runs\five_seeds\per_seed_metrics.csv and C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTrainingPaddedBox\runs\five_seeds\seed_summary.csv


Show where each seed prediction CSV lives and confirm the shared validation sequence IDs.


In [3]:
# Cell purpose: Show where each seed prediction CSV lives and confirm the shared validation sequence IDs.
# Inputs: The completed five-seed experiment and its fixed split manifest.
# Outputs: Printed file paths and validation IDs for manual comparison.

# Each best checkpoint also has its own complete per-box validation CSV.
# Run each seed independently; train_one_seed reuses only a completed matching run.
for seed in SEEDS:
    print(seed, OUTPUT / f'seed_{seed}' / 'validation_predictions.csv')
print('Same validation IDs for every run:', SPLIT['val_sequences'])

42 C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTrainingPaddedBox\runs\five_seeds\seed_42\validation_predictions.csv
123 C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTrainingPaddedBox\runs\five_seeds\seed_123\validation_predictions.csv
2026 C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTrainingPaddedBox\runs\five_seeds\seed_2026\validation_predictions.csv
7 C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTrainingPaddedBox\runs\five_seeds\seed_7\validation_predictions.csv
99 C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTrainingPaddedBox\runs\five_seeds\seed_99\validation_predictions.csv
Same validation IDs for every run: ['seq_72fdfff4b4353a42', 'seq_1c1d4f5e71a2f554', 'seq_f7ee32f1dd417802', 'seq_210721b0d576149f', 'seq_fa2eee6269a5dad3', 'seq_d749b853cf795b6a', 'seq_0dc2e3d874c804fc', 'seq_de542b63e6b19bc2', 'seq_f9043534

Run this stage using this experiment's configuration and caches.
